### Import packegues

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib as mpl
from matplotlib.ticker import AutoLocator, AutoMinorLocator, LogLocator
import glob
import cv2
import os
import re

In [ ]:
data_directory = '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/shadow_test/shadow_test'
R = 1170000.0 # cm
r_sphere_radius = R # cm
r_sphere_center_cm = ( 1425937.5, 1425937.5, 1425937.5 ) # cm
r_sphere_radius_index = ( 32.0, 32.0, 32.0 ) # cells
r_shadowing_sphere_radius = 1.5*R/4.5 # cm
r_shadowing_sphere_center_cm = ( 6120562.5, 1425937.5, 1425937.5 ) # cm
r_shadowing_sphere_radius_index = ( 139.0, 32.0, 32.0 ) # cells

mesh_files = glob.glob(os.path.join(data_directory, 'mesh_plt*.h5'))
mesh_files_nums = [filenames.split('mesh_plt')[1].split('.h5')[0] for filenames in mesh_files]
mesh_files_sorted = [f for _, f in sorted(zip(mesh_files_nums, mesh_files))]
mesh_files_sorted = np.array(mesh_files_sorted)
nfiles = len(mesh_files_sorted)
print(f'Found {len(mesh_files_sorted)} mesh files.')

In [ ]:
rho_ye_T_h5py = h5py.File(data_directory+'/rho_Ye_T.hdf5', 'r')

# number of cells
Nx = np.array(rho_ye_T_h5py['/ncellsx'])
Ny = np.array(rho_ye_T_h5py['/ncellsy'])
Nz = np.array(rho_ye_T_h5py['/ncellsz'])
print(f'Nx = {Nx}, Ny = {Ny}, Nz = {Nz}')

# cell size
dx = ( np.array(rho_ye_T_h5py['/xmax']) - np.array(rho_ye_T_h5py['/xmin']) ) / np.array(rho_ye_T_h5py['/ncellsx']) # cm
dy = ( np.array(rho_ye_T_h5py['/ymax']) - np.array(rho_ye_T_h5py['/ymin']) ) / np.array(rho_ye_T_h5py['/ncellsy']) # cm
dz = ( np.array(rho_ye_T_h5py['/zmax']) - np.array(rho_ye_T_h5py['/zmin']) ) / np.array(rho_ye_T_h5py['/ncellsz']) # cm
print(f'dx = {dx} cm, dy = {dy} cm, dz = {dz} cm')

# cell centers
xc = np.linspace(dx / 2, dx * (Nx - 0.5), Nx) # cm
yc = np.linspace(dy / 2, dy * (Ny - 0.5), Ny) # cm
zc = np.linspace(dz / 2, dz * (Nz - 0.5), Nz) # cm

# cell centers mesh
Xc, Yc, Zc = np.meshgrid(xc, yc, zc, indexing='ij') # cm

bh_radius = 5.43e+05 # cm
bh_center_x = 48.0e+5 # cm
bh_center_y = 48.0e+5 # cm
bh_center_z = 16.0e+5 # cm

distance_from_bh = np.sqrt( (Xc - bh_center_x)**2 + (Yc - bh_center_y)**2 + (Zc - bh_center_z)**2 )
# mask_bh = (distance_from_bh < bh_radius + 1*np.max([dx,dy,dz]))
mask_bh = (distance_from_bh < bh_radius)
mask_x = (Xc < dx) | (Xc > (Nx - 1) * dx)
mask_y = (Yc < dy) | (Yc > (Ny - 1) * dy)
mask_z = (Zc < dz) | (Zc > (Nz - 1) * dz)
mask = mask_bh | mask_x | mask_y | mask_z

rho_ye_T_h5py.close()

In [ ]:
# Font settings
mpl.rcParams['font.size'] = 22
mpl.rcParams['font.family'] = 'serif'
mpl.rc('text', usetex=False)

# Tick settings
mpl.rcParams['xtick.major.width'] = 2
mpl.rcParams['xtick.major.pad'] = 8
mpl.rcParams['xtick.minor.size'] = 4

mpl.rcParams['xtick.minor.width'] = 2
mpl.rcParams['ytick.major.size'] = 7
mpl.rcParams['ytick.major.width'] = 2
mpl.rcParams['ytick.minor.size'] = 4
mpl.rcParams['ytick.minor.width'] = 2

# Axis linewidth
mpl.rcParams['axes.linewidth'] = 2

# Tick direction and enabling ticks on all sides
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'
mpl.rcParams['xtick.top'] = True
mpl.rcParams['ytick.right'] = True

# Function to apply custom tick locators and other settings to an Axes object
def apply_custom_settings(ax, log_scale_y=False):

    if log_scale_y:
        # Use LogLocator for the y-axis if it's in log scale
        ax.set_yscale('log')
        ax.yaxis.set_major_locator(LogLocator(base=10.0))
        ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs='auto', numticks=100))
    else:
        # Use AutoLocator for regular scales
        ax.yaxis.set_major_locator(AutoLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
    
    # Apply the AutoLocator for the x-axis
    ax.xaxis.set_major_locator(AutoLocator())
    ax.xaxis.set_minor_locator(AutoMinorLocator())

def plot_color_map(x, y, z, bh_r, bh_x, bh_y, min_cb, max_cb, x_label, y_label, title, cbar_label, colormap, filename, doshow=True, dosave=True):

    fig, ax = plt.subplots(figsize=(12, 8))

    # Plot pcolormesh
    # Set non-data points (NaN) to black in the colormap
    cmap = plt.get_cmap(colormap).copy()
    cmap.set_bad(color='black')

    # Plot pcolormesh with masked array to handle NaNs
    c = ax.pcolormesh(x, y, z, shading='auto', cmap=cmap, vmin=min_cb, vmax=max_cb)
    
    # Add contour lines
    # contour = ax.contour(x, y, z, colors='black', linewidths=1.5, levels=3)
    # contour = ax.contour(x, y, z, colors='black', linewidths=1.5, levels=[32.0, 32.2, 32.4])
    # ax.clabel(contour, inline=True, fontsize=15, fmt='%1.1f')

    # Plot settings
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    # ax.set_title(title+'\nmin: {:.2e}\nmax: {:.2e}'.format(np.nanmin(z), np.nanmax(z)))
    ax.set_title(title)

    linescolors = 'black'
    linesalpha = 0.5

    # Emitter and absorber spheres. Centers and radii are stored in cm;
    # the plot axes are in km. Both spheres have y = z, so (x, y) is also
    # the correct (x, z) center on the xz slices.
    emitter_circle = plt.Circle(
        (r_sphere_center_cm[0] / 1e5, r_sphere_center_cm[1] / 1e5),
        radius=r_sphere_radius / 1e5,
        edgecolor=linescolors,
        facecolor='none',
        linewidth=2,
        label='emitter',
        zorder=5,
        alpha=linesalpha,
    )
    absorber_circle = plt.Circle(
        (r_shadowing_sphere_center_cm[0] / 1e5, r_shadowing_sphere_center_cm[1] / 1e5),
        radius=r_shadowing_sphere_radius / 1e5,
        edgecolor=linescolors,
        facecolor='none',
        linewidth=2,
        label='absorber',
        zorder=5,
        alpha=linesalpha,
    )
    ax.add_patch(emitter_circle)
    ax.add_patch(absorber_circle)
    # ax.legend(loc='upper right')

    # External tangents: one contact point on each sphere, extended to +x.
    emitter_center_km = np.array([
        r_sphere_center_cm[0] / 1e5,
        r_sphere_center_cm[1] / 1e5,
    ])
    absorber_center_km = np.array([
        r_shadowing_sphere_center_cm[0] / 1e5,
        r_shadowing_sphere_center_cm[1] / 1e5,
    ])
    emitter_radius_km = r_sphere_radius / 1e5
    absorber_radius_km = r_shadowing_sphere_radius / 1e5
    center_separation = absorber_center_km - emitter_center_km
    separation = np.linalg.norm(center_separation)
    line_angle = np.arctan2(center_separation[1], center_separation[0])
    tangent_tilt = np.arcsin((emitter_radius_km - absorber_radius_km) / separation)
    x_domain_max = np.nanmax(x)
    for side in (+1, -1):  # upper, then lower
        radius_angle = line_angle + side * (np.pi / 2 - tangent_tilt)
        radius_direction = np.array([np.cos(radius_angle), np.sin(radius_angle)])
        emitter_touch = emitter_center_km + emitter_radius_km * radius_direction
        absorber_touch = absorber_center_km + absorber_radius_km * radius_direction
        tangent_direction = absorber_touch - emitter_touch
        extension = (x_domain_max - emitter_touch[0]) / tangent_direction[0]
        tangent_end = emitter_touch + extension * tangent_direction
        ax.plot(
            [emitter_touch[0], tangent_end[0]],
            [emitter_touch[1], tangent_end[1]],
            color=linescolors,
            linewidth=2,
            zorder=6,
            alpha=linesalpha,
        )

    # Internal tangents: opposite contact sides, extended to +x.
    internal_tilt = np.arcsin((emitter_radius_km + absorber_radius_km) / separation)
    for side in (+1, -1):
        radius_angle = line_angle + side * (np.pi / 2 - internal_tilt)
        radius_direction = np.array([np.cos(radius_angle), np.sin(radius_angle)])
        emitter_touch = emitter_center_km + emitter_radius_km * radius_direction
        absorber_touch = absorber_center_km - absorber_radius_km * radius_direction
        tangent_direction = absorber_touch - emitter_touch
        extension = (x_domain_max - emitter_touch[0]) / tangent_direction[0]
        tangent_end = emitter_touch + extension * tangent_direction
        ax.plot(
            [emitter_touch[0], tangent_end[0]],
            [emitter_touch[1], tangent_end[1]],
            color=linescolors,
            linewidth=2,
            zorder=6,
            alpha=linesalpha,
        )

    # Add color bar
    cbar = fig.colorbar(c, ax=ax, label=cbar_label)
    cbar.ax.yaxis.set_minor_locator(AutoMinorLocator())

    apply_custom_settings(ax, False)

    # Ensure equal aspect ratio
    ax.set_aspect('equal', 'box')

    # Save figure
    if dosave:
        fig.savefig(filename, format='png', bbox_inches='tight')

    # Display figure
    if doshow:
        plt.show()
    # display(fig)
    
    # Close figure
    plt.close(fig)

def plot_color_vector_map(x, y, z, bh_r, bh_x, bh_y, vecx, vecy, ve, min_cb, max_cb, x_label, y_label, title, cbar_label, colormap, filename, doshow=True, dosave=True):

    fig, ax = plt.subplots(figsize=(12, 8))


    # Plot pcolormesh
    # Set non-data points (NaN) to black in the colormap
    cmap = plt.get_cmap(colormap).copy()
    cmap.set_bad(color='black')

    # Plot pcolormesh with masked array to handle NaNs
    c = ax.pcolormesh(x, y, z, shading='auto', cmap=cmap, vmin=min_cb, vmax=max_cb)
    
    ax.quiver(x[::ve,::ve], y[::ve,::ve], ve*vecx[::ve,::ve], ve*vecy[::ve,::ve], color='black', scale=1, scale_units='xy', angles='xy')

    # Add contour lines
    # contour = ax.contour(x, y, z, colors='black', linewidths=1.5, levels=3)
    contour = ax.contour(x, y, z, colors='black', linewidths=1.5, levels=[-1.4,-1.0,-0.6])
    ax.clabel(contour, inline=True, fontsize=15, fmt='%1.1f')

    # Plot settings
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    # ax.set_title(title+'\nmin: {:.2e}\nmax: {:.2e}'.format(np.nanmin(z), np.nanmax(z)))
    ax.set_title(title)

    # # Plot a black circle at the black hole center
    # circle = plt.Circle(
    #     (bh_x, bh_y),  # center in km
    #     radius=bh_r,  # radius in km
    #     color='black',
    #     fill=True,
    #     linewidth=2
    # )
    # ax.add_artist(circle)

    # Add color bar
    cbar = fig.colorbar(c, ax=ax, label=cbar_label)
    cbar.ax.yaxis.set_minor_locator(AutoMinorLocator())

    apply_custom_settings(ax, False)

    # Ensure equal aspect ratio
    ax.set_aspect('equal', 'box')

    # Save figure
    if dosave:
        fig.savefig(filename, format='png', bbox_inches='tight')

    # Display figure
    if doshow:
        plt.show()
    # display(fig)
    
    # Close figure
    plt.close(fig)

def plot_vector_map(zup, zlow, vector_x_up, vector_y_up, vector_x_low, vector_y_low, time_ms, colorbarlabel, zindx, yindx, filname, minbar, maxbar, doshow=True, dosave=True):

    plot_color_vector_map(  x=Xc[1:-1,1:-1,zindx]/1e5, # km
                            y=Yc[1:-1,1:-1,zindx]/1e5, # km
                            z=zup,
                            bh_r=6.8, # km
                            bh_x=48.0, # km
                            bh_y=48.0, # km
                            vecx=vector_x_up,
                            vecy=vector_y_up,
                            ve=4,
                            min_cb=minbar,
                            max_cb=maxbar, 
                            x_label=r'$x \, (\mathrm{km})$', 
                            y_label=r'$y \, (\mathrm{km})$', 
                            title=fr'$z={Zc[0,0,zindx]/1e5:.1f}\,\mathrm{{km}}$, $t = {time_ms:.2f}\,\mathrm{{ms}}$', 
                            cbar_label=colorbarlabel, 
                            colormap='viridis',
                            filename=f'{filname}_xy.png',
                            doshow=doshow,
                            dosave=dosave)

    plot_color_vector_map(  x=Xc[1:-1,yindx,1:-1]/1e5, 
                            y=Zc[1:-1,yindx,1:-1]/1e5, 
                            z=zlow,
                            bh_r=6.8, # km
                            bh_x=48.0, # km
                            bh_y=16.0, # km
                            vecx=vector_x_low,
                            vecy=vector_y_low,
                            ve=4, 
                            min_cb=minbar,
                            max_cb=maxbar, 
                            x_label=r'$x \, (\mathrm{km})$', 
                            y_label=r'$z \, (\mathrm{km})$', 
                            title=rf'$y={Yc[0,yindx,0]/1e5:.1f}\,\mathrm{{km}}$, $t = {time_ms:.2f}\,\mathrm{{ms}}$', 
                            cbar_label=colorbarlabel, 
                            colormap='viridis',
                            filename=f'{filname}_xz.png',
                            doshow=doshow,
                            dosave=dosave)
    
def plot_up_low(zup, zlow, time_ms, colorbarlabel, zindx, yindx, filname, minbar, maxbar, doshow=True, dosave=True):

    plot_color_map( x=Xc[1:-1,1:-1,zindx]/1e5, # km 
                    y=Yc[1:-1,1:-1,zindx]/1e5, # km 
                    z=zup, 
                    bh_r=6.8, # km
                    bh_x=48.0, # km
                    bh_y=48.0, # km
                    min_cb=minbar,
                    max_cb=maxbar,                    
                    # min_cb=2.27e1,
                    # max_cb=2.55e1,
                    # min_cb=2.22e1,
                    # max_cb=2.69e1,
                    # min_cb=None,
                    # max_cb=None, 
                    x_label=r'$x \, (\mathrm{km})$',
                    y_label=r'$y \, (\mathrm{km})$',
                    title=fr'$z={Zc[0,0,zindx]/1e5:.1f}\,\mathrm{{km}}$, $t = {time_ms:.5f}\,\mathrm{{ms}}$', 
                    cbar_label=colorbarlabel, 
                    colormap='rainbow',
                    filename=f'{filname}_xy.png',
                    doshow=doshow,
                    dosave=dosave)

    plot_color_map( x=Xc[1:-1,yindx,1:-1]/1e5, # km
                    y=Zc[1:-1,yindx,1:-1]/1e5, # km
                    z=zlow,
                    bh_r=6.8, # km
                    bh_x=48.0, # km
                    bh_y=16.0, # km
                    min_cb=minbar,
                    max_cb=maxbar,
                    # min_cb=2.27e1,
                    # max_cb=2.55e1,
                    # min_cb=2.22e1,
                    # max_cb=2.69e1, 
                    # min_cb=None,
                    # max_cb=None, 
                    x_label=r'$x \, (\mathrm{km})$', 
                    y_label=r'$z \, (\mathrm{km})$', 
                    title=rf'$y={Yc[0,yindx,0]/1e5:.1f}\,\mathrm{{km}}$, $t = {time_ms:.5f}\,\mathrm{{ms}}$', 
                    cbar_label=colorbarlabel, 
                    colormap='rainbow',
                    filename=f'{filname}_xz.png',
                    doshow=doshow,
                    dosave=dosave)
    
def plot_up_low_seismic(zup, zlow, time_ms, colorbarlabel, zindx, yindx, filname, doshow=True, dosave=True):

    plot_color_map( x=Xc[:,:,zindx]/1e5, # km 
                    y=Yc[:,:,zindx]/1e5, # km 
                    z=zup,
                    min_cb=np.nanmax(zup)*-1,
                    max_cb=np.nanmax(zup), 
                    x_label=r'$x \, (\mathrm{km})$',
                    y_label=r'$y \, (\mathrm{km})$',
                    title=fr'$z={Zc[0,0,zindx]/1e5:.1f}\,\mathrm{{km}}$, $t = {time_ms:.2f}\,\mathrm{{ms}}$', 
                    cbar_label=colorbarlabel, 
                    colormap='seismic',
                    filename=f'{filname}_xy.png',
                    doshow=doshow,
                    dosave=dosave)

    plot_color_map( x=Xc[:,yindx,:]/1e5, # km
                    y=Zc[:,yindx,:]/1e5, # km
                    z=zlow,
                    min_cb=np.nanmax(zup)*-1,
                    max_cb=np.nanmax(zup), 
                    x_label=r'$x \, (\mathrm{km})$', 
                    y_label=r'$z \, (\mathrm{km})$', 
                    title=rf'$y={Yc[0,yindx,0]/1e5:.1f}\,\mathrm{{km}}$, $t = {time_ms:.2f}\,\mathrm{{ms}}$', 
                    cbar_label=colorbarlabel, 
                    colormap='seismic',
                    filename=f'{filname}_xz.png',
                    doshow=doshow,
                    dosave=dosave)
    
def plot_color_two_vector_map(x, y, z, vecx1, vecy1, vecx2, vecy2, ve, min_cb, max_cb, x_label, y_label, title, cbar_label, colormap, filename, doshow=True, dosave=True):

    fig, ax = plt.subplots(figsize=(12, 8))

    # Plot pcolormesh
    c = ax.pcolormesh(x, y, z, shading='auto', cmap=colormap, vmin=min_cb, vmax=max_cb)
    
    vecscale = 1.0

    ax.quiver(x[::ve,::ve], y[::ve,::ve], vecscale*ve*vecx2[::ve,::ve], vecscale*ve*vecy2[::ve,::ve], color='white', scale=1.0, scale_units='xy', angles='xy')
    ax.quiver(x[::ve,::ve], y[::ve,::ve], vecscale*ve*vecx1[::ve,::ve], vecscale*ve*vecy1[::ve,::ve], color='black', scale=1.0, scale_units='xy', angles='xy')

    # Add contour lines
    # contour = ax.contour(x, y, z, colors='black', linewidths=1.0, levels=3)
    # ax.clabel(contour, inline=True, fontsize=15, fmt='%1.1e')
    # contour = ax.contour(x, y, z, colors='black', linewidths=0.5, levels=[24, 25, 26])
    # ax.clabel(contour, inline=True, fontsize=15, fmt='%d')

    # Plot settings
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    # ax.set_title(title+'\nmin: {:.2e}\nmax: {:.2e}'.format(np.nanmin(z), np.nanmax(z)))
    ax.set_title(title)

    # Add color bar
    cbar = fig.colorbar(c, ax=ax, label=cbar_label)
    cbar.ax.yaxis.set_minor_locator(AutoMinorLocator())

    apply_custom_settings(ax, False)

    # Ensure equal aspect ratio
    ax.set_aspect('equal', 'box')

    # Save figure
    if dosave:
        fig.savefig(filename, format='png', bbox_inches='tight')

    # Display figure
    if doshow:
        plt.show()
    # display(fig)
    
    # Close figure
    plt.close(fig)

def plot_two_vector_map(zup, zlow, vector_x1_up, vector_y1_up, vector_x2_up, vector_y2_up, vector_x1_low, vector_y1_low, vector_x2_low, vector_y2_low, time_ms, colorbarlabel, zindx, yindx, filname, minbar, maxbar, doshow=True, dosave=True):

    ven = 4

    plot_color_two_vector_map(  x=Xc[:,:,zindx]/1e5, # km
                                y=Yc[:,:,zindx]/1e5, # km
                                z=zup,
                                vecx1=vector_x1_up,
                                vecy1=vector_y1_up,
                                vecx2=vector_x2_up,
                                vecy2=vector_y2_up,
                                ve=ven,
                                min_cb=minbar,
                                max_cb=maxbar, 
                                x_label=r'$x \, (\mathrm{km})$', 
                                y_label=r'$y \, (\mathrm{km})$', 
                                title=fr'$z={Zc[0,0,zindx]/1e5:.1f}\,\mathrm{{km}}$, $t = {time_ms:.2f}\,\mathrm{{ms}}$', 
                                cbar_label=colorbarlabel, 
                                colormap='viridis',
                                filename=f'{filname}_xy.png',
                                doshow=doshow,
                                dosave=dosave)

    plot_color_two_vector_map(  x=Xc[:,yindx,:]/1e5, 
                                y=Zc[:,yindx,:]/1e5, 
                                z=zlow,
                                vecx1=vector_x1_low,
                                vecy1=vector_y1_low,
                                vecx2=vector_x2_low,
                                vecy2=vector_y2_low,
                                ve=ven, 
                                min_cb=minbar,
                                max_cb=maxbar, 
                                x_label=r'$x \, (\mathrm{km})$', 
                                y_label=r'$z \, (\mathrm{km})$', 
                                title=rf'$y={Yc[0,yindx,0]/1e5:.1f}\,\mathrm{{km}}$, $t = {time_ms:.2f}\,\mathrm{{ms}}$', 
                                cbar_label=colorbarlabel, 
                                colormap='viridis',
                                filename=f'{filname}_xz.png',
                                doshow=doshow,
                                dosave=dosave)
    

In [ ]:
x_slice_indx = 96 # cell number
y_slice_indx = 32 # cell number
z_slice_indx = 32 # cell number
time_indx    = -1 # last time step

# for time_indx in [8,12,16,20,24,28,32,36,40]:
# for time_indx in [1,2,3,4,5,6,7,8,9,10]:

for filepath in [mesh_files_sorted[-1]]:
# for filepath in mesh_files_sorted:

    print(f'Processing file: {filepath}')

    alldata_h5 = h5py.File(filepath, 'r')
    alldata_dict = {key: np.array(alldata_h5[key]) for key in alldata_h5.keys()}
    alldata_h5.close()
    alldata = alldata_dict.copy()
    print("Keys in alldata_dict:", list(alldata.keys()))

    # if time_indx==0:
    if True:
        doshow = True
    else:
        doshow = False

    time_ms = np.array(alldata['t(s)']) * 1.0e3 # ms

    # global min: -14.43554531020161 global max: 24.104102117542855
    max_cbar = 24.104102117542855
    min_cbar = -14.43554531020161
    # min_cbar = -30
    # max_cbar = 2e22
    # min_cbar = 1e15
    # max_cbar = None
    # min_cbar = None

    ###############################################
    # Neutrino diagonal terms
    ###############################################

    # # Electron neutrino number density
    zup = np.where(mask[1:-1, 1:-1, z_slice_indx], np.nan, np.array(alldata['N00_Re(1|ccm)'])[1:-1, 1:-1, z_slice_indx])
    zlow = np.where(mask[1:-1, y_slice_indx, 1:-1], np.nan, np.array(alldata['N00_Re(1|ccm)'])[1:-1, y_slice_indx, 1:-1])
    zup = np.log10(zup)
    zlow = np.log10(zlow)
    # zup = zup
    # zlow = zlow
    print("global min:", np.nanmin([np.nanmin(zup), np.nanmin(zlow)]), "global max:", np.nanmax([np.nanmax(zup), np.nanmax(zlow)]))
    colorbarlabel = fr'$\log ( \, n_{{\nu_{{e}}}}\,[\mathrm{{cm}}^{{-3}}]\,)$'
    filename=f'./plots/{filepath.split('mesh_plt')[1].split('.h5')[0]}_nee'
    mincbar=min_cbar
    maxcbar=max_cbar
    # mincbar=None
    # maxcbar=None
    plot_up_low(zup, zlow, time_ms, colorbarlabel, z_slice_indx, y_slice_indx, filename, mincbar,maxcbar, doshow=doshow, dosave=True)

    # with h5py.File('./plots/n_e.h5', 'w') as f:
    #     f.create_dataset('xup', data=Xc[1:-1,1:-1,16]/1e5)
    #     f.create_dataset('yup', data=Yc[1:-1,1:-1,16]/1e5)
    #     f.create_dataset('zup', data=zup)
    #     f.create_dataset('xlow', data=Xc[1:-1,48,1:-1]/1e5)
    #     f.create_dataset('ylow', data=Zc[1:-1,48,1:-1]/1e5)
    #     f.create_dataset('zlow', data=zlow)

    # # Muon neutrino number density
    # zup = np.where(mask[1:-1, 1:-1, z_slice_indx], np.nan, np.array(alldata['N11_Re(1|ccm)'])[1:-1, 1:-1, z_slice_indx])
    # zlow = np.where(mask[1:-1, y_slice_indx, 1:-1], np.nan, np.array(alldata['N11_Re(1|ccm)'])[1:-1, y_slice_indx, 1:-1])
    # zup = np.log10(zup)
    # zlow = np.log10(zlow)
    # colorbarlabel = fr'$\log ( \, n_{{\nu_{{\mu}}}}\,[\mathrm{{cm}}^{{-3}}]\,)$'
    # filename=f'./plots/{filepath.split('mesh_plt')[1].split('.h5')[0]}_nuu'
    # mincbar=None
    # maxcbar=None
    # plot_up_low(zup, zlow, time_ms, colorbarlabel, z_slice_indx, y_slice_indx, filename, mincbar,maxcbar, doshow=doshow, dosave=True)

    # with h5py.File('./plots/n_x.h5', 'w') as f:
    #     f.create_dataset('xup', data=Xc[1:-1,1:-1,16]/1e5)
    #     f.create_dataset('yup', data=Yc[1:-1,1:-1,16]/1e5)
    #     f.create_dataset('zup', data=zup)
    #     f.create_dataset('xlow', data=Xc[1:-1,48,1:-1]/1e5)
    #     f.create_dataset('ylow', data=Zc[1:-1,48,1:-1]/1e5)
    #     f.create_dataset('zlow', data=zlow)

    # # Tauon neutrino number density
    # zup = np.where(mask[:, :, z_slice_indx], np.nan, np.array(alldata['N22_Re(1|ccm)'])[time_indx, :, :, z_slice_indx])
    # zlow = np.where(mask[:, y_slice_indx, :], np.nan, np.array(alldata['N22_Re(1|ccm)'])[time_indx, :, y_slice_indx, :])
    # zup = np.log10(zup)
    # zlow = np.log10(zlow)
    # colorbarlabel = fr'$\log ( \, n_{{\nu_{{\tau}}}}\,[\mathrm{{cm}}^{{-3}}]\,)$'
    # filename=f'./plots/{finaldir}_ntt_t_{time_indx}'
    # mincbar=None
    # maxcbar=None
    # plot_up_low(zup, zlow, time_ms, colorbarlabel, z_slice_indx, y_slice_indx, filename, mincbar,maxcbar, doshow=doshow, dosave=True)

    ###############################################
    # Antineutrino diagonal terms
    ###############################################

    # # Electron antineutrino number density
    # zup = np.where(mask[1:-1, 1:-1, z_slice_indx], np.nan, np.array(alldata['N00_Rebar(1|ccm)'])[1:-1, 1:-1, z_slice_indx])
    # zlow = np.where(mask[1:-1, y_slice_indx, 1:-1], np.nan, np.array(alldata['N00_Rebar(1|ccm)'])[1:-1, y_slice_indx, 1:-1])
    # zup = np.log10(zup)
    # zlow = np.log10(zlow)
    # print("global min:", np.nanmin([np.nanmin(zup), np.nanmin(zlow)]), "global max:", np.nanmax([np.nanmax(zup), np.nanmax(zlow)]))
    # colorbarlabel = fr'$\log ( \, n_{{\bar{{\nu}}_{{e}}}}\,[\mathrm{{cm}}^{{-3}}]\,)$'
    # filename=f'./plots/{filepath.split('mesh_plt')[1].split('.h5')[0]}_nbaree'
    # mincbar=min_cbar
    # maxcbar=max_cbar
    # mincbar=None
    # maxcbar=None
    # plot_up_low(zup, zlow, time_ms, colorbarlabel, z_slice_indx, y_slice_indx, filename, mincbar,maxcbar, doshow=doshow, dosave=True)

    # with h5py.File('./plots/nbar_e.h5', 'w') as f:
    #     f.create_dataset('xup', data=Xc[1:-1,1:-1,16]/1e5)
    #     f.create_dataset('yup', data=Yc[1:-1,1:-1,16]/1e5)
    #     f.create_dataset('zup', data=zup)
    #     f.create_dataset('xlow', data=Xc[1:-1,48,1:-1]/1e5)
    #     f.create_dataset('ylow', data=Zc[1:-1,48,1:-1]/1e5)
    #     f.create_dataset('zlow', data=zlow)

    # # Muon antineutrino number density
    # zup = np.where(mask[:, :, z_slice_indx], np.nan, np.array(alldata['N11_Rebar(1|ccm)'])[time_indx, :, :, z_slice_indx])
    # zlow = np.where(mask[:, y_slice_indx, :], np.nan, np.array(alldata['N11_Rebar(1|ccm)'])[time_indx, :, y_slice_indx, :])
    # zup = np.log10(zup)
    # zlow = np.log10(zlow)
    # colorbarlabel = fr'$\log ( \, n_{{\bar{{\nu}}_{{\mu}}}}\,[\mathrm{{cm}}^{{-3}}]\,)$'
    # filename=f'./plots/{finaldir}_nbaruu_t_{time_indx}'
    # mincbar=None
    # maxcbar=None
    # plot_up_low(zup, zlow, time_ms, colorbarlabel, z_slice_indx, y_slice_indx, filename, mincbar,maxcbar, doshow=doshow, dosave=True)

    # # Tauon antineutrino number density
    # zup = np.where(mask[:, :, z_slice_indx], np.nan, np.array(alldata['N22_Rebar(1|ccm)'])[time_indx, :, :, z_slice_indx])
    # zlow = np.where(mask[:, y_slice_indx, :], np.nan, np.array(alldata['N22_Rebar(1|ccm)'])[time_indx, :, y_slice_indx, :])
    # zup = np.log10(zup)
    # zlow = np.log10(zlow)
    # colorbarlabel = fr'$\log ( \, n_{{\bar{{\nu}}_{{\tau}}}}\,[\mathrm{{cm}}^{{-3}}]\,)$'
    # filename=f'./plots/{finaldir}_nbartt_t_{time_indx}'
    # mincbar=None
    # maxcbar=None
    # plot_up_low(zup, zlow, time_ms, colorbarlabel, z_slice_indx, y_slice_indx, filename, mincbar,maxcbar, doshow=doshow, dosave=True)

    ###############################################
    # All heavies neutrinos number density toguether
    ###############################################

    # # Heavy neutrino number density
    # zup = np.where(mask[1:-1, 1:-1, z_slice_indx], np.nan, np.array(alldata['N11_Re(1|ccm)'])[time_indx, 1:-1, 1:-1, z_slice_indx])
    # zlow = np.where(mask[1:-1, y_slice_indx, 1:-1], np.nan, np.array(alldata['N11_Re(1|ccm)'])[time_indx, 1:-1, y_slice_indx, 1:-1])
    # zup = np.log10(zup)
    # zlow = np.log10(zlow)
    # print("global min:", np.nanmin([np.nanmin(zup), np.nanmin(zlow)]), "global max:", np.nanmax([np.nanmax(zup), np.nanmax(zlow)]))
    # colorbarlabel = fr'$\log ( \, n_{{ \, \nu_{{x}} }} \, [\mathrm{{cm}}^{{-3}}] \, )$'
    # filename=f'./plots/{finaldir}_nxx_t_{time_indx}'
    # mincbar=min_cbar
    # maxcbar=max_cbar
    # plot_up_low(zup, zlow, time_ms, colorbarlabel, z_slice_indx, y_slice_indx, filename, mincbar,maxcbar, doshow=doshow, dosave=True)


